## Multi-stage builds & BuildKit

**Multi-stage builds** solve a common problem: compiled languages (Go, Rust, Java, C++) need a toolchain to *build* but not to *run*. You don't want `gcc` or the JDK shipping in production. The trick: multiple `FROM` blocks in one Dockerfile — the final image is only the *last* stage; earlier stages are scratch pads.

```dockerfile
# ----- builder stage -----
FROM golang:1.23 AS builder
WORKDIR /src
COPY . .
RUN CGO_ENABLED=0 go build -o /out/server ./cmd/server

# ----- runtime stage -----
FROM gcr.io/distroless/static:nonroot
COPY --from=builder /out/server /server
USER nonroot:nonroot
ENTRYPOINT ["/server"]
```

Stage 1 uses the fat `golang` image (compiler, source, dep cache) and builds the binary. Stage 2 starts fresh from tiny `distroless/static` (a few MB, no shell, no package manager) and `COPY --from=builder` pulls in **just the compiled binary**. A Go service that's 1.5 GB with the SDK becomes ~15 MB — faster pulls, fewer CVEs, no toolchain to attack. The pattern applies anywhere a "build" phase is distinct from a "run" phase (Node: build in stage 1, copy `dist/` to a slim stage).

**BuildKit** is the modern build backend (default on Docker 23+ and Desktop). It upgrades the engine well beyond the original sequential builder:

- **Parallelism** — independent stages build at the same time.
- **Cache mounts** — `RUN --mount=type=cache,target=/root/.cache/pip pip install …` reuses pip's cache across builds without baking it into a layer.
- **Build secrets** — `RUN --mount=type=secret,id=tok …` reads a secret without committing it to a layer (`docker build --secret …`).
- **Cross-platform** — build `linux/amd64` and `linux/arm64` from one Dockerfile on one machine.

The CLI that drives it is **`docker buildx`**, now the default builder — most of the time `docker build` and `docker buildx build` are interchangeable. Together, multi-stage + BuildKit are how you get small, fast, reproducible images — the payoff of everything in this module.